In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install --upgrade pip
!pip install xarray netCDF4 geopandas shapely

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [netCDF4]


In [10]:
import sys
import os
import glob
import xarray as xr

PROJECT_PATH = "/content/drive/MyDrive/cmip6-madagascar-projection"
sys.path.insert(0, PROJECT_PATH)

from src.config import *
from src.data_loader import *
from src.preprocessing import *
from src.regridding import *
from src.masking import *

In [3]:
RAW_PATH = os.path.join(PROJECT_PATH, "data/raw")
REGRID_PATH = os.path.join(PROJECT_PATH, "data/regridded")
PROCESSED_PATH = os.path.join(PROJECT_PATH, "data/processed")
BOUNDARY_FULL_PATH = os.path.join(PROJECT_PATH, "data/boundaries")

os.makedirs(REGRID_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [4]:
africa_shape = load_africa_boundary(BOUNDARY_FULL_PATH)

In [5]:
target_lat, target_lon = create_target_grid(TARGET_RESOLUTION)

In [6]:
models = [
    "CNRM-CM6-1",
    "MPI-ESM1-2-LR",
    "IPSL-CM6A-LR"
]

variables = [
    "ts",
    "pr"
]

experiments = [
    "historical",
    "ssp245",
    "ssp585"
]

In [7]:
def find_file(model, variable, experiment):

    pattern = os.path.join(
        RAW_PATH,
        f"{variable}_Amon_{model}_{experiment}_*.nc"
    )

    files = glob.glob(pattern)

    if len(files) == 0:
        print(f"Fichier manquant : {model} {variable} {experiment}")
        return None

    return files[0]

In [9]:
drop_vars = ["time_bounds", "lat_bounds", "lon_bounds", "bnds"]

for model in models:

    print("\n==========================")
    print("MODEL :", model)
    print("==========================")

    for experiment in experiments:

        print("Experiment :", experiment)

        for variable in variables:

            file_path = find_file(model, variable, experiment)

            if file_path is None:
                continue

            print("Processing :", model, variable, experiment)

            ds = xr.open_dataset(file_path)

            # subset Afrique
            ds = subset_bbox(ds, AFRICA_BBOX)

            # supprimer variables inutiles
            for var in drop_vars:
                if var in ds.variables:
                    ds = ds.drop_vars(var)

            # regrid
            ds_rg = regrid_dataset(ds, target_lat, target_lon)

            # tri grille
            ds_rg = ds_rg.sortby("lat")
            ds_rg = ds_rg.sortby("lon")

            # mask Afrique
            ds_rg = apply_africa_mask(ds_rg, africa_shape)

            # sauvegarde
            out_name = f"{model}_{variable}_{experiment}_regridded.nc"

            out_path = os.path.join(REGRID_PATH, out_name)

            ds_rg.to_netcdf(out_path)

            print("Saved :", out_name)

print("\nTRAITEMENT TERMINE")


MODEL : CNRM-CM6-1
Experiment : historical
Processing : CNRM-CM6-1 ts historical
Saved : CNRM-CM6-1_ts_historical_regridded.nc
Processing : CNRM-CM6-1 pr historical
Saved : CNRM-CM6-1_pr_historical_regridded.nc
Experiment : ssp245
Processing : CNRM-CM6-1 ts ssp245
Saved : CNRM-CM6-1_ts_ssp245_regridded.nc
Processing : CNRM-CM6-1 pr ssp245
Saved : CNRM-CM6-1_pr_ssp245_regridded.nc
Experiment : ssp585
Processing : CNRM-CM6-1 ts ssp585
Saved : CNRM-CM6-1_ts_ssp585_regridded.nc
Processing : CNRM-CM6-1 pr ssp585
Saved : CNRM-CM6-1_pr_ssp585_regridded.nc

MODEL : MPI-ESM1-2-LR
Experiment : historical
Processing : MPI-ESM1-2-LR ts historical
Saved : MPI-ESM1-2-LR_ts_historical_regridded.nc
Processing : MPI-ESM1-2-LR pr historical
Saved : MPI-ESM1-2-LR_pr_historical_regridded.nc
Experiment : ssp245
Processing : MPI-ESM1-2-LR ts ssp245
Saved : MPI-ESM1-2-LR_ts_ssp245_regridded.nc
Processing : MPI-ESM1-2-LR pr ssp245
Saved : MPI-ESM1-2-LR_pr_ssp245_regridded.nc
Experiment : ssp585
Processing : 